## Dataset profiling (List15 + List16)

This notebook:
- Loads every provided dataset in `Other_Analysis/` (both `.csv` and `.xlsx` files)
- Audits schema, missingness, duplicates, and basic distributions
- Identifies join keys for integration (`REG_NO`, `SEMESTER_INDEX`, `ACADEMIC_YEAR`, `SEMESTER`)

**Excel files** such as `students_list15.xlsx`, `students_list16.xlsx`, `dim_date_2022_2026.xlsx`, `academic_progression_list15.xlsx`, `academic_progression_list16.xlsx`, `course_catalog_ucu.xlsx`, and `student_grades_summary_*.xlsx` are imported and profiled when present. Multi-sheet workbooks are loaded with each sheet profiled separately (e.g. `student_grades_summary_list15` → sheets `SEMESTER_GPA`, `STUDENT_CGPA`).

**When the same data exists as both CSV and Excel**, we **prefer the CSV** for profiling (and document the choice in the "CSV vs Excel overlap" section below).

In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR

csv_files = sorted(DATA_DIR.glob("*.csv"))
xlsx_files = sorted(DATA_DIR.glob("*.xlsx"))

print("CSV files:", len(csv_files))
print("Excel files:", len(xlsx_files))
(csv_files, xlsx_files)

CSV files: 18
Excel files: 8


([WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/fact_student_academic_performance_list15.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/fact_student_academic_performance_list16.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/faculties_departments.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/high_schools_dimension.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/high_schools_list.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/student_attendance_list15.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/student_attendance_list16.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/student_grades_list15.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/student_grades_list16.csv'),
  WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/student_high_schools_all.csv'),
  Wi

In [ ]:
def load_csv(path: Path) -> pd.DataFrame:
    # Keep parsing conservative; we coerce dates later per-table.
    return pd.read_csv(path)


def load_xlsx(path: Path) -> dict[str, pd.DataFrame]:
    """Load an Excel file. Returns a dict name -> DataFrame (one entry per sheet). Requires openpyxl."""
    out = {}
    xl = pd.ExcelFile(path, engine="openpyxl")
    stem = path.stem
    for sheet_name in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name=sheet_name)
        # Single-sheet workbook: use stem (e.g. students_list15). Multi-sheet: stem_sheetname.
        key = stem if len(xl.sheet_names) == 1 else f"{stem}_{sheet_name}"
        out[key] = df
    return out


def profile_df(df: pd.DataFrame, name: str, key_cols: list[str] | None = None) -> dict:
    info = {
        "name": name,
        "rows": len(df),
        "cols": len(df.columns),
        "null_cells": int(df.isna().sum().sum()),
        "dup_rows": int(df.duplicated().sum()),
    }

    if key_cols:
        existing = [c for c in key_cols if c in df.columns]
        if len(existing) == len(key_cols):
            info["dup_keys"] = int(df.duplicated(existing).sum())
            info["null_keys"] = int(df[existing].isna().any(axis=1).sum())
        else:
            info["dup_keys"] = None
            info["null_keys"] = None

    return info


def value_counts_safe(df: pd.DataFrame, col: str, n: int = 10) -> pd.Series:
    if col not in df.columns:
        return pd.Series(dtype="object")
    return df[col].value_counts(dropna=False).head(n)


datasets = {}
dataset_source = {}  # name -> ("csv"|"xlsx", path_or_note)
skipped_xlsx_overlap = []  # (xlsx_path, dataset_name) when CSV was chosen instead

for p in csv_files:
    datasets[p.stem] = load_csv(p)
    dataset_source[p.stem] = ("csv", str(p.name))

for p in xlsx_files:
    for name, df in load_xlsx(p).items():
        if name in datasets:
            skipped_xlsx_overlap.append((str(p.name), name))
        else:
            datasets[name] = df
            dataset_source[name] = ("xlsx", str(p.name))

list(datasets.keys())

['fact_student_academic_performance_list15',
 'fact_student_academic_performance_list16',
 'faculties_departments',
 'high_schools_dimension',
 'high_schools_list',
 'student_attendance_list15',
 'student_attendance_list16',
 'student_grades_list15',
 'student_grades_list16',
 'student_high_schools_all',
 'student_high_schools_list15',
 'student_high_schools_list16',
 'student_payments_list15',
 'student_payments_list16',
 'student_sponsorships_list15',
 'student_sponsorships_list16',
 'student_transcript_list15',
 'student_transcript_list16',
 'academic_progression_list15',
 'academic_progression_list16',
 'course_catalog_ucu',
 'dim_date_2022_2026',
 'student_grades_summary_list15_SEMESTER_GPA',
 'student_grades_summary_list15_STUDENT_CGPA',
 'student_grades_summary_list16_SEMESTER_GPA',
 'student_grades_summary_list16_STUDENT_CGPA',
 'students_list15',
 'students_list16']

### CSV vs Excel: overlap and choice

When a dataset exists as **both** a `.csv` and a `.xlsx` file (same logical name, e.g. `course_catalog_ucu`), we **use the CSV** for profiling. CSVs are preferred for consistency, version control, and scripted pipelines. The table below shows any such overlaps and confirms the chosen source. Excel-only datasets (e.g. `students_list15`, `dim_date_2022_2026`, `academic_progression_list15/16`, `course_catalog_ucu` when no CSV exists) are loaded from the `.xlsx` files and profiled as usual.

In [ ]:
# Overlap: same dataset available as CSV and Excel → we chose CSV
csv_stems = {p.stem for p in csv_files}
overlap = [(xlsx_path, name) for xlsx_path, name in skipped_xlsx_overlap]

if overlap:
    overlap_df = pd.DataFrame(overlap, columns=["Excel_file_skipped", "dataset_name"])
    overlap_df["chosen_source"] = "CSV"
    overlap_df["csv_file"] = overlap_df["dataset_name"].map(lambda n: next((p.name for p in csv_files if p.stem == n), ""))
    print("When both CSV and Excel exist for the same dataset, we use the CSV:\n")
    display(overlap_df[["dataset_name", "csv_file", "Excel_file_skipped", "chosen_source"]])
else:
    print("No overlap: no dataset had both a CSV and an Excel file with the same name.")

# Excel-only: datasets that exist only as .xlsx (no CSV counterpart)
xlsx_only = [name for name, (src, _) in dataset_source.items() if src == "xlsx"]
print("\nExcel-only datasets (profiled from .xlsx):")
pd.Series(sorted(xlsx_only), name="dataset_name").to_frame()

No overlap: no dataset had both a CSV and an Excel file with the same name.

Excel-only datasets (profiled from .xlsx):


,dataset_name
0,academic_progression_list15
1,academic_progression_list16
2,course_catalog_ucu
3,dim_date_2022_2026
4,student_grades_summary_list15_SEMESTER_GPA
5,student_grades_summary_list15_STUDENT_CGPA
6,student_grades_summary_list16_SEMESTER_GPA
7,student_grades_summary_list16_STUDENT_CGPA
8,students_list15
9,students_list16


In [ ]:
KEY_MAP = {
    # CSV tables
    "student_transcript_list15": ["REG_NO", "SEMESTER_INDEX"],
    "student_transcript_list16": ["REG_NO", "SEMESTER_INDEX"],
    "fact_student_academic_performance_list15": ["REG_NO", "SEMESTER_INDEX"],
    "fact_student_academic_performance_list16": ["REG_NO", "SEMESTER_INDEX"],
    "student_grades_list15": ["REG_NO", "SEMESTER_INDEX", "COURSE_CODE"],
    "student_grades_list16": ["REG_NO", "SEMESTER_INDEX", "COURSE_CODE"],
    "student_attendance_list15": ["REG_NO", "DATE"],
    "student_attendance_list16": ["REG_NO", "DATE"],
    "student_payments_list15": ["PAYMENT_ID"],
    "student_payments_list16": ["PAYMENT_ID"],
    "student_sponsorships_list15": ["SPONSOR_PAYMENT_ID"],
    "student_sponsorships_list16": ["SPONSOR_PAYMENT_ID"],
    "student_high_schools_all": ["REG_NO"],
    # Excel: students list (one row per student)
    "students_list15": ["REG_NO"],
    "students_list16": ["REG_NO"],
    # Excel: grades summary sheets
    "student_grades_summary_list15_SEMESTER_GPA": ["REG_NO", "SEMESTER_INDEX"],
    "student_grades_summary_list16_SEMESTER_GPA": ["REG_NO", "SEMESTER_INDEX"],
    "student_grades_summary_list15_STUDENT_CGPA": ["REG_NO"],
    "student_grades_summary_list16_STUDENT_CGPA": ["REG_NO"],
    # Excel: academic progression, course catalog, date dimension
    "academic_progression_list15": ["PROGRESSION_ID"],
    "academic_progression_list16": ["PROGRESSION_ID"],
    "course_catalog_ucu": ["PROGRAM", "SEMESTER_INDEX", "COURSE_CODE"],
    "dim_date_2022_2026": ["date_key"],
}

profiles = []
for name, df in datasets.items():
    profiles.append(profile_df(df, name, KEY_MAP.get(name)))

profiles_df = pd.DataFrame(profiles).sort_values(["rows"], ascending=False)
profiles_df

,name,rows,cols,null_cells,dup_rows,dup_keys,null_keys
6,student_attendance_list16,2126474,5,0,0,0.0,0.0
5,student_attendance_list15,1078445,5,0,0,0.0,0.0
19,academic_progression_list16,229921,8,0,0,0.0,0.0
8,student_grades_list16,228259,17,247941,0,0.0,0.0
13,student_payments_list16,122566,15,183846,0,0.0,0.0
18,academic_progression_list15,121666,8,0,0,0.0,0.0
7,student_grades_list15,120091,17,130511,0,0.0,0.0
12,student_payments_list15,64836,15,97329,0,0.0,0.0
1,fact_student_academic_performance_list16,34929,15,0,0,0.0,0.0
17,student_transcript_list16,34929,21,0,0,0.0,0.0


## Quick categorical checks

These checks help validate assumptions for later integration (e.g., attendance status values, payment status values, sponsorship status values).

In [ ]:
checks = {
    "attendance_STATUS": ("student_attendance_list15", "STATUS"),
    "payments_PAYMENT_STATUS": ("student_payments_list15", "PAYMENT_STATUS"),
    "sponsorship_STATUS": ("student_sponsorships_list15", "STATUS"),
    "grades_STATUS": ("student_grades_list15", "STATUS"),
}

for label, (ds, col) in checks.items():
    if ds in datasets:
        print("\n==", label, "==")
        print(value_counts_safe(datasets[ds], col, n=20))


== attendance_STATUS ==
STATUS
PRESENT    1024492
LATE         32531
ABSENT       21422
Name: count, dtype: int64

== payments_PAYMENT_STATUS ==
PAYMENT_STATUS
SUCCESS    52002
FAILED     12834
Name: count, dtype: int64

== sponsorship_STATUS ==
STATUS
APPROVED    2988
PENDING     1022
Name: count, dtype: int64

== grades_STATUS ==
STATUS
Completed    92497
FEX          17174
FCW           6895
MEX           3525
Name: count, dtype: int64


## Save profiling summary

This writes a small CSV report (rows/cols/nulls/duplicates) that other notebooks can reuse.

In [ ]:
out_dir = BASE_DIR / "outputs"
out_dir.mkdir(exist_ok=True)

profiles_df.to_csv(out_dir / "dataset_profiles.csv", index=False)
profiles_df

,name,rows,cols,null_cells,dup_rows,dup_keys,null_keys
6,student_attendance_list16,2126474,5,0,0,0.0,0.0
5,student_attendance_list15,1078445,5,0,0,0.0,0.0
19,academic_progression_list16,229921,8,0,0,0.0,0.0
8,student_grades_list16,228259,17,247941,0,0.0,0.0
13,student_payments_list16,122566,15,183846,0,0.0,0.0
18,academic_progression_list15,121666,8,0,0,0.0,0.0
7,student_grades_list15,120091,17,130511,0,0.0,0.0
12,student_payments_list15,64836,15,97329,0,0.0,0.0
1,fact_student_academic_performance_list16,34929,15,0,0,0.0,0.0
17,student_transcript_list16,34929,21,0,0,0.0,0.0


## Deeper per-dataset summaries

These sections produce the core descriptive analytics that later inform feature engineering and modeling.

In [ ]:
trans = pd.concat([
    datasets["student_transcript_list15"],
    datasets["student_transcript_list16"],
], ignore_index=True)

print("Rows:", len(trans))
print("Students:", trans["REG_NO"].nunique())
print("Semesters:", trans["SEMESTER_INDEX"].nunique())
print("CGPA describe:\n", trans["CGPA"].describe())

# CGPA by semester index
cgpa_by_sem = trans.groupby("SEMESTER_INDEX")["CGPA"].agg(["count","mean","median","std","min","max"]).reset_index()
cgpa_by_sem.head(20)

Rows: 53457
Students: 9929
Semesters: 14
CGPA describe:
 count    53457.000000
mean         3.263431
std          0.628582
min          1.500000
25%          2.810000
50%          3.280000
75%          3.720000
max          5.000000
Name: CGPA, dtype: float64


,SEMESTER_INDEX,count,mean,median,std,min,max
0,1,9992,3.239420,3.24,0.714236,1.50,5.00
1,2,9811,3.242346,3.26,0.643292,1.50,5.00
2,3,8152,3.264709,3.28,0.616665,1.66,4.95
3,4,6124,3.257867,3.28,0.607005,1.72,4.94
4,5,5193,3.268317,3.29,0.598683,1.76,4.79
5,6,4507,3.290515,3.32,0.590538,1.77,4.81
6,7,2271,3.302004,3.33,0.574835,1.80,4.69
7,8,1882,3.311562,3.33,0.570654,1.85,4.67
8,9,1787,3.314645,3.34,0.564354,1.86,4.69
9,10,1025,3.302263,3.33,0.568199,1.89,4.67


In [ ]:
att = pd.concat([
    datasets["student_attendance_list15"],
    datasets["student_attendance_list16"],
], ignore_index=True)

att["STATUS"] = att["STATUS"].astype(str).str.upper().str.strip()
att_status = att["STATUS"].value_counts(dropna=False)
att_status

STATUS
PRESENT    3045039
LATE         95907
ABSENT       63973
Name: count, dtype: int64

In [ ]:
pay = pd.concat([
    datasets["student_payments_list15"],
    datasets["student_payments_list16"],
], ignore_index=True)

pay["PAYMENT_STATUS"] = pay["PAYMENT_STATUS"].astype(str).str.upper().str.strip()
pay["AMOUNT_UGX"] = pd.to_numeric(pay["AMOUNT_UGX"], errors="coerce")

pay.groupby("PAYMENT_STATUS").agg(
    txns=("PAYMENT_ID","count"),
    students=("REG_NO","nunique"),
    amount_sum=("AMOUNT_UGX","sum"),
    amount_mean=("AMOUNT_UGX","mean"),
).sort_values("txns", ascending=False)

,txns,students,amount_sum,amount_mean
PAYMENT_STATUS,,,,
SUCCESS,150155,9928,130739804354,870698.973421
FAILED,37247,9121,32436398681,870845.938760


In [ ]:
spon = pd.concat([
    datasets["student_sponsorships_list15"],
    datasets["student_sponsorships_list16"],
], ignore_index=True)

spon["STATUS"] = spon["STATUS"].astype(str).str.upper().str.strip()
spon["AMOUNT_SPONSORED_UGX"] = pd.to_numeric(spon["AMOUNT_SPONSORED_UGX"], errors="coerce")

spon_summary = spon.groupby("STATUS").agg(
    events=("SPONSOR_PAYMENT_ID","count"),
    students=("REG_NO","nunique"),
    amount_sum=("AMOUNT_SPONSORED_UGX","sum"),
    amount_mean=("AMOUNT_SPONSORED_UGX","mean"),
).sort_values("events", ascending=False)

spon_summary

,events,students,amount_sum,amount_mean
STATUS,,,,
APPROVED,8192,2026,14982217024,1.828884e+06
PENDING,2752,1464,4998006489,1.816136e+06


In [ ]:
grades = pd.concat([
    datasets["student_grades_list15"],
    datasets["student_grades_list16"],
], ignore_index=True)

grades["FINAL_MARK_100"] = pd.to_numeric(grades["FINAL_MARK_100"], errors="coerce")
grades["COURSE_UNITS"] = pd.to_numeric(grades["COURSE_UNITS"], errors="coerce")

# Pass/fail quick view
status_counts = grades["STATUS"].astype(str).str.upper().str.strip().value_counts(dropna=False)
status_counts

STATUS
COMPLETED    273598
FEX           44650
FCW           19668
MEX           10434
Name: count, dtype: int64